# NALAPRO Project — Part 1
## Simple Neural Network Classifier on 20-Newsgroups
### Word2Vec · TF-IDF · GloVe

---

## Tools used
| Tool | Purpose |
|------|---------|
| `scikit-learn` | Dataset, TF-IDF vectoriser, evaluation metrics |
| `nltk` | Tokenisation, stop-word removal, lemmatisation |
| `gensim` | Word2Vec training and GloVe loading |
| `PyTorch` | Neural network and training loop |
| `matplotlib / seaborn` | Visualisations |
| `Claude (Anthropic)` | Scaffolding and assistance with code |

**Reference:** *Hands-On Large Language Models*, Jay Alammar & Maarten Grootendorst  
→ https://github.com/HandsOnLLM/Hands-On-Large-Language-Models  
**Word2Vec deep dive:** Jay Alammar, *The Illustrated Word2vec*  
→ https://jalammar.github.io/illustrated-word2vec/  
**In-class notebooks:** `pytorch_solution.ipynb`, `NLP_Pipeline.ipynb`

---


## Setup & reproducibility


In [ ]:
# ── install if not already present ──────────────────────────────────────────────
%pip install -q torch gensim scikit-learn nltk matplotlib seaborn wandb tqdm


---
## Experiment tracking (in-notebook)

Rather than an external experiment-tracking server, all metrics are captured directly in the
`history` dictionary returned by `train_model()` and displayed as tables and plots inside
this notebook.

**Per-epoch signals tracked during training**

| Signal | Description |
|--------|-------------|
| `train_loss` | Cross-entropy loss averaged over each mini-batch |
| `val_loss` | Cross-entropy loss on the full validation set |
| `val_acc` | Fraction of validation examples correctly classified |
| `weight_norm` | L2 norm of all weight matrices — how far weights have moved from initialisation |
| `bias_norm` | L2 norm of all bias vectors |
| `grad_norm` | L2 norm of all gradients after the final backward pass of each epoch |

In [ ]:
import os, random, math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── Reproducibility ──────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Hardware ─────────────────────────────────────────────────────────────────────
DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("Running on:", DEVICE)


In [ ]:
# ── Google Drive setup (Google Colab only) ────────────────────────────────────────
# Mounts your Drive so that CSVs and plots from all four parts share one persistent
# folder: MyDrive/NALAPRO/. On a local machine this cell is a no-op.
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    NALAPRO_DIR = '/content/drive/MyDrive/NALAPRO'
    os.makedirs(NALAPRO_DIR, exist_ok=True)
    os.chdir(NALAPRO_DIR)
    _IN_COLAB = True
    print(f'Google Colab detected.')
    print(f'Working directory → {NALAPRO_DIR}')
    print('All CSVs and plots will persist to Google Drive → MyDrive/NALAPRO/')
except ModuleNotFoundError:
    _IN_COLAB = False
    print(f'Not in Colab. Working directory: {os.getcwd()}')

---
## 1.a · Data loading & preprocessing

### The 20-Newsgroups dataset
`fetch_20newsgroups` is ~18 000 Usenet posts spread over 20 topic groups
(e.g. `comp.graphics`, `rec.sport.hockey`, `soc.religion.christian`).
scikit-learn provides a pre-built train/test split.

### Preprocessing pipeline

The pipeline mirrors what we built in `NLP_Pipeline.ipynb` in class:

```
Raw text
  ↓  lower-case everything
  ↓  tokenise with a regex that keeps only alphabetic strings
  ↓  remove stop-words  (very common words like "the", "and" add noise)
  ↓  lemmatise          (reduce inflected forms: "running" → "run")
  ↓  keep tokens with len > 2  (drop "of", "it", etc.)
List of clean tokens
```

In [ ]:
from sklearn.datasets import fetch_20newsgroups

REMOVE = ("headers", "footers", "quotes")
ng_train = fetch_20newsgroups(subset="train", remove=REMOVE, random_state=SEED)
ng_test  = fetch_20newsgroups(subset="test",  remove=REMOVE, random_state=SEED)

class_names = ng_train.target_names
NUM_CLASSES  = len(class_names)

print(f"Train documents : {len(ng_train.data):>6,}")
print(f"Test documents  : {len(ng_test.data):>6,}")
print(f"Number of classes: {NUM_CLASSES}")
print("\nClasses:")
for i, c in enumerate(class_names):
    print(f"  {i:2d}. {c}")


In [ ]:
# Download the NLTK data packages we need (runs once, silent)
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import RegexpTokenizer
from nltk.corpus   import stopwords as sw
from nltk.stem     import WordNetLemmatizer

STOPWORDS  = set(sw.words("english"))
LEMMATIZER = WordNetLemmatizer()
# [a-zA-Z]+ matches only alphabetic strings — no numbers, punctuation, or URLs
TOKENIZER  = RegexpTokenizer(r"[a-zA-Z]+")

def preprocess(text: str) -> list[str]:
    """
    Convert raw Usenet post to a list of clean tokens.
    Steps: lower-case → tokenise → remove stop-words → lemmatise → length filter.
    """
    tokens = TOKENIZER.tokenize(text.lower())
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return tokens

# ── Sanity check on the first document ───────────────────────────────────────────
print("Raw  (first 300 chars):", ng_train.data[0][:300])
print("\nClean tokens (first 20):", preprocess(ng_train.data[0])[:20])


In [ ]:
from tqdm.auto import tqdm

print("Tokenising training documents…")
train_tokens = [preprocess(t) for t in tqdm(ng_train.data)]
print("Tokenising test documents…")
test_tokens  = [preprocess(t) for t in tqdm(ng_test.data)]

y_train_full = np.array(ng_train.target)
y_test       = np.array(ng_test.target)

# ── Carve out a validation split (10 %) ──────────────────────────────────────────
# We use stratify= to ensure every class appears in both splits at the same ratio.
# This prevents accidental over- or under-representation of rare classes in validation.
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    np.arange(len(train_tokens)), test_size=0.10,
    stratify=y_train_full, random_state=SEED,
)
train_tok = [train_tokens[i] for i in train_idx]
val_tok   = [train_tokens[i] for i in val_idx]
y_train   = y_train_full[train_idx]
y_val     = y_train_full[val_idx]

print(f"\nFinal split — train: {len(train_tok):,}  val: {len(val_tok):,}  test: {len(test_tokens):,}")


---
## The neural network (shared across all experiments)

The architecture is fixed by the assignment: **two fully-connected layers with a ReLU
non-linearity in between** — exactly the network from `pytorch_solution.ipynb`.

In [ ]:
class SimpleNN(nn.Module):
    """
    Two-layer feed-forward network as specified in the project:
        Linear(input_dim, hidden_dim) → ReLU → Linear(hidden_dim, output_dim)

    Arguments:
        input_dim  – dimensionality of the input representation
        hidden_dim – width of the hidden layer  (hyper-parameter)
        output_dim – number of classes (20 for 20-Newsgroups)
    """
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, hidden_dim)
        self.relu    = nn.ReLU()
        self.linear2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch_size, input_dim)
        x = self.linear1(x)   # → (batch_size, hidden_dim)
        x = self.relu(x)      # zero out negatives
        x = self.linear2(x)   # → (batch_size, 20) — raw logits
        return x


In [ ]:
def train_model(model, X_train, y_train, X_val, y_val,
                epochs=20, batch_size=128, lr=1e-3, name='run'):
    """
    Generic training loop used for every Part-1 experiment.
    Returns a history dict with per-epoch:
        train_loss, val_loss, val_acc, weight_norm, bias_norm, grad_norm.
    """
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    X_tr_t = torch.tensor(X_train, dtype=torch.float32)
    y_tr_t = torch.tensor(y_train, dtype=torch.long)
    X_va_t = torch.tensor(X_val,   dtype=torch.float32).to(DEVICE)
    y_va_t = torch.tensor(y_val,   dtype=torch.long).to(DEVICE)

    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                        batch_size=batch_size, shuffle=True)

    history = {
        'train_loss': [], 'val_loss': [], 'val_acc': [],
        'weight_norm': [], 'bias_norm': [], 'grad_norm': [],
    }

    for epoch in range(1, epochs + 1):
        # ── Training ────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= len(loader.dataset)

        # ── Validation ──────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            v_logits = model(X_va_t)
            v_loss   = criterion(v_logits, y_va_t).item()
            v_acc    = (v_logits.argmax(1) == y_va_t).float().mean().item()

        # ── Parameter & gradient norms ────────────────────────────────────
        # weight_norm: combined L2 norm of all weight matrices (not bias vectors)
        # bias_norm:   combined L2 norm of all bias vectors
        # grad_norm:   combined L2 norm of all gradients (weights + biases)
        w_norm = sum(
            p.data.norm(2).item() ** 2
            for n, p in model.named_parameters() if 'weight' in n
        ) ** 0.5
        b_norm = sum(
            p.data.norm(2).item() ** 2
            for n, p in model.named_parameters() if 'bias' in n
        ) ** 0.5
        g_norm = sum(
            p.grad.norm(2).item() ** 2
            for p in model.parameters()
            if p.requires_grad and p.grad is not None
        ) ** 0.5

        history['train_loss'].append(epoch_loss)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        history['weight_norm'].append(w_norm)
        history['bias_norm'].append(b_norm)
        history['grad_norm'].append(g_norm)

        if epoch % 5 == 0 or epoch == 1:
            print(f'[{name}] epoch {epoch:3d} | '
                  f'train_loss={epoch_loss:.4f}  val_loss={v_loss:.4f}  val_acc={v_acc:.4f}')
    return history


def evaluate_on_test(model, X_test, y_test, name='run'):
    """Report accuracy and macro-F1 on the held-out test set."""
    from sklearn.metrics import accuracy_score, f1_score
    model.eval()
    X_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        preds = model(X_t).argmax(1).cpu().numpy()
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='macro')
    print(f'[{name}]  test_acc={acc:.4f}   test_macro_f1={f1:.4f}')
    return {'acc': acc, 'f1': f1, 'preds': preds}


---
## 1.b · Word2Vec embeddings

**Reference:** Jay Alammar, *The Illustrated Word2vec* — https://jalammar.github.io/illustrated-word2vec/  

In [ ]:
from gensim.models import Word2Vec

W2V_DIM     = 100   # embedding size — same as GloVe below for a fair comparison
W2V_WINDOW  = 5     # context window: how many words left/right count as "context"
W2V_MIN_CNT = 5     # ignore words that appear fewer than 5 times (too noisy to embed)

# ── 1 epoch ──────────────────────────────────────────────────────────────────────
print("Training Word2Vec — 1 epoch …")
w2v_1ep = Word2Vec(
    sentences=train_tok,
    vector_size=W2V_DIM, window=W2V_WINDOW, min_count=W2V_MIN_CNT,
    workers=4, epochs=1, seed=SEED,
)

# ── 20 epochs ────────────────────────────────────────────────────────────────────
print("Training Word2Vec — 20 epochs …")
w2v_20ep = Word2Vec(
    sentences=train_tok,
    vector_size=W2V_DIM, window=W2V_WINDOW, min_count=W2V_MIN_CNT,
    workers=4, epochs=20, seed=SEED,
)

print(f"Vocabulary size: {len(w2v_1ep.wv):,} words")


### Visualising the embedding space with t-SNE

**t-SNE** (t-Distributed Stochastic Neighbour Embedding) takes 100-dimensional word vectors
and squashes them into 2D while trying to preserve *neighbourhood structure*:
words that were close in 100D should still be close in 2D.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Representative words from 6 of the 20 newsgroups
probe_words = [
    # comp.* groups
    "computer", "software", "windows", "graphics", "hardware", "monitor",
    # rec.sport.*
    "baseball", "hockey", "team", "player", "season", "game",
    # sci.space
    "space", "orbit", "satellite", "nasa", "rocket", "mission",
    # soc.religion + alt.atheism
    "god", "jesus", "religion", "christian", "atheism", "faith",
    # rec.autos + rec.motorcycles
    "car", "engine", "bike", "motorcycle", "drive",
    # sci.crypt
    "encryption", "security", "key", "algorithm",
]
# Keep only words present in BOTH models (because min_count may differ between runs)
probe_words = [w for w in probe_words if w in w2v_1ep.wv and w in w2v_20ep.wv]
print(f"Probing {len(probe_words)} words:", probe_words)

vec_1ep  = np.stack([w2v_1ep.wv[w]  for w in probe_words])
vec_20ep = np.stack([w2v_20ep.wv[w] for w in probe_words])

# perplexity is a t-SNE hyperparameter related to the effective number of neighbours.
# Values between 5–30 work well for small probe sets.
def tsne_2d(vecs, seed=SEED):
    return TSNE(n_components=2, perplexity=8, random_state=seed, init="pca",
                learning_rate="auto").fit_transform(vecs)

proj_1ep  = tsne_2d(vec_1ep)
proj_20ep = tsne_2d(vec_20ep)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, proj, title in [(axes[0], proj_1ep,  "Word2Vec — 1 epoch"),
                        (axes[1], proj_20ep, "Word2Vec — 20 epochs")]:
    ax.scatter(proj[:, 0], proj[:, 1], s=30, alpha=0.8)
    for i, w in enumerate(probe_words):
        ax.annotate(w, (proj[i, 0], proj[i, 1]), fontsize=9, alpha=0.9)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")

plt.suptitle("Word vector geometry: does more training help?", fontsize=15)
plt.tight_layout()
plt.savefig("part1_tsne_w2v.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part1_tsne_w2v.png  (include this in the report)")


In [ ]:
# ── Mean-pool word vectors into document vectors ──────────────────────────────────
def mean_pool(token_lists, model):
    """
    For each document (list of tokens), look up each word's vector and average them.
    Documents with NO known words stay as the zero vector — this rarely happens
    after removing min_count=5 words, but is handled gracefully.
    """
    vecs = np.zeros((len(token_lists), model.wv.vector_size), dtype=np.float32)
    for i, toks in enumerate(token_lists):
        word_vecs = [model.wv[t] for t in toks if t in model.wv]
        if word_vecs:
            vecs[i] = np.mean(word_vecs, axis=0)
    return vecs

# Build feature matrices for both W2V variants
X_tr_1ep   = mean_pool(train_tok, w2v_1ep);   X_va_1ep  = mean_pool(val_tok, w2v_1ep)
X_te_1ep   = mean_pool(test_tokens, w2v_1ep)
X_tr_20ep  = mean_pool(train_tok, w2v_20ep);  X_va_20ep = mean_pool(val_tok, w2v_20ep)
X_te_20ep  = mean_pool(test_tokens, w2v_20ep)

print("Document vector shapes:", X_tr_1ep.shape, "(train), then val, test – same columns")


In [ ]:
# ── Experiment 1 : W2V 1 epoch ────────────────────────────────────────────────────
HIDDEN = 256; EPOCHS = 30

model_b1 = SimpleNN(W2V_DIM, HIDDEN, NUM_CLASSES)
hist_b1  = train_model(model_b1, X_tr_1ep, y_train, X_va_1ep, y_val,
                        epochs=EPOCHS, name='w2v_1ep')
res_b1   = evaluate_on_test(model_b1, X_te_1ep, y_test, name='w2v_1ep')


In [ ]:
# ── Experiment 2 : W2V 20 epochs ─────────────────────────────────────────────────
model_b20 = SimpleNN(W2V_DIM, HIDDEN, NUM_CLASSES)
hist_b20  = train_model(model_b20, X_tr_20ep, y_train, X_va_20ep, y_val,
                         epochs=EPOCHS, name='w2v_20ep')
res_b20   = evaluate_on_test(model_b20, X_te_20ep, y_test, name='w2v_20ep')


---
## 1.c · TF-IDF

### Why does TF-IDF often outperform Word2Vec on 20-Newsgroups?

Because 20-Newsgroups classification is **keyword-driven**.  
The word `"hockey"` is almost exclusively in hockey posts. TF-IDF gives it extreme weight.  
Word2Vec *averages* hundreds of word vectors, so the clear signal from `"hockey"` is diluted
by the background noise of hundreds of other tokens.  
TF-IDF is essentially a *lossless* representation of which discriminative words appeared.

**Limitation:** TF-IDF sees `"car"` and `"automobile"` as completely different dimensions —
it has no concept of synonymy. This matters less for 20-Newsgroups (topics use consistent
terminology) than it would for, say, a paraphrase-detection task.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Join clean tokens back to space-separated strings for the sklearn vectoriser
train_text_str = [" ".join(t) for t in train_tok]
val_text_str   = [" ".join(t) for t in val_tok]
test_text_str  = [" ".join(t) for t in test_tokens]

TFIDF_VOCAB = 20_000   # keep the 20k most informative terms

# fit_transform on TRAINING data only — then transform val/test.
# Fitting on val/test would be data leakage.
tfidf = TfidfVectorizer(max_features=TFIDF_VOCAB, sublinear_tf=True)
X_tr_tfidf = tfidf.fit_transform(train_text_str).toarray().astype(np.float32)
X_va_tfidf = tfidf.transform(val_text_str).toarray().astype(np.float32)
X_te_tfidf = tfidf.transform(test_text_str).toarray().astype(np.float32)

print("TF-IDF matrix shape:", X_tr_tfidf.shape)
print("Example: top-10 terms by weight in document 0:")
doc0 = X_tr_tfidf[0]
top10 = np.argsort(doc0)[::-1][:10]
vocab_list = np.array(tfidf.get_feature_names_out())
for idx in top10:
    if doc0[idx] > 0:
        print(f"  {vocab_list[idx]:20s}  {doc0[idx]:.4f}")


In [ ]:
# ── Experiment 3 : TF-IDF ────────────────────────────────────────────────────────
model_c = SimpleNN(TFIDF_VOCAB, HIDDEN, NUM_CLASSES)
hist_c  = train_model(model_c, X_tr_tfidf, y_train, X_va_tfidf, y_val,
                       epochs=EPOCHS, name='tfidf')
res_c   = evaluate_on_test(model_c, X_te_tfidf, y_test, name='tfidf')


---
## 1.d · Additional experiment — Pre-trained GloVe embeddings

### Hypothesis
Word2Vec trained on ~11k newsgroup posts gives noisy embeddings because the corpus is small —
there simply is not enough text to see many word-pair co-occurrences.  
**GloVe-wiki-gigaword-100** was trained on *6 billion tokens* from Wikipedia + Gigaword,
which means even infrequent technical words are embedded reliably.

**Prediction:** Pre-trained GloVe → better mean-pooled document vectors → higher classification accuracy,
*without touching the neural network architecture at all*.


In [ ]:
import gensim.downloader as gensim_api

# This downloads ~130 MB on first call; cached afterwards
print("Loading GloVe-wiki-gigaword-100 (may take a minute on first run)…")
glove = gensim_api.load("glove-wiki-gigaword-100")
print(f"GloVe vocabulary size: {len(glove.key_to_index):,}")
print("Example — most similar to 'hockey':",
      [w for w, _ in glove.most_similar("hockey", topn=5)])


In [ ]:
def mean_pool_kv(token_lists, kv):
    """Same as mean_pool() but works with a gensim KeyedVectors object (e.g. GloVe)."""
    vecs = np.zeros((len(token_lists), kv.vector_size), dtype=np.float32)
    for i, toks in enumerate(token_lists):
        word_vecs = [kv[t] for t in toks if t in kv]
        if word_vecs:
            vecs[i] = np.mean(word_vecs, axis=0)
    return vecs

X_tr_glv = mean_pool_kv(train_tok, glove)
X_va_glv = mean_pool_kv(val_tok,   glove)
X_te_glv = mean_pool_kv(test_tokens, glove)
print("GloVe document vectors:", X_tr_glv.shape)


In [ ]:
# ── Experiment 4 : GloVe ─────────────────────────────────────────────────────────
model_d = SimpleNN(100, HIDDEN, NUM_CLASSES)
hist_d  = train_model(model_d, X_tr_glv, y_train, X_va_glv, y_val,
                       epochs=EPOCHS, name='glove')
res_d   = evaluate_on_test(model_d, X_te_glv, y_test, name='glove')


---
## Final comparison & analysis

In [ ]:
import pandas as pd

def count_params(model):
    n_w = sum(p.numel() for n, p in model.named_parameters() if 'weight' in n)
    n_b = sum(p.numel() for n, p in model.named_parameters() if 'bias'   in n)
    return n_w, n_b

rows = []
for label, model, hist, res, input_dim in [
    ('W2V 1 epoch',       model_b1,  hist_b1,  res_b1,  W2V_DIM),
    ('W2V 20 epochs',     model_b20, hist_b20, res_b20, W2V_DIM),
    ('TF-IDF (20k)',      model_c,   hist_c,   res_c,   TFIDF_VOCAB),
    ('GloVe pre-trained', model_d,   hist_d,   res_d,   100),
]:
    n_w, n_b = count_params(model)
    rows.append({
        'Experiment':    label,
        'Input dim':     f'{input_dim:,}',
        'Hidden':        HIDDEN,
        'Weight params': f'{n_w:,}',
        'Bias params':   f'{n_b:,}',
        'Total params':  f'{n_w + n_b:,}',
        'Weight L2':     f'{hist["weight_norm"][-1]:.3f}',
        'Bias L2':       f'{hist["bias_norm"][-1]:.3f}',
        'Grad L2':       f'{hist["grad_norm"][-1]:.3f}',
        'Test Acc':      f'{res["acc"]:.4f}',
        'Macro-F1':      f'{res["f1"]:.4f}',
    })

df = pd.DataFrame(rows)
print('─' * 105)
print('EXPERIMENT SUMMARY — PARAMETERS, WEIGHT NORMS & PERFORMANCE')
print('─' * 105)
print(df.to_string(index=False))
df.to_csv('part1_results.csv', index=False)
print('\n→ Saved part1_results.csv')


In [ ]:
# ── Per-layer weight & bias statistics ────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

model_items = [
    ('W2V 1ep',  model_b1),
    ('W2V 20ep', model_b20),
    ('TF-IDF',   model_c),
    ('GloVe',    model_d),
]

layer_rows = []
for label, model in model_items:
    for pname, param in model.named_parameters():
        data = param.data.cpu().numpy()
        layer_rows.append({
            'Experiment': label,
            'Layer':      pname,
            'Shape':      str(tuple(data.shape)),
            'Params':     f'{data.size:,}',
            'Mean':       f'{data.mean():.5f}',
            'Std':        f'{data.std():.5f}',
            'Min':        f'{data.min():.5f}',
            'Max':        f'{data.max():.5f}',
            'L2 norm':    f'{np.linalg.norm(data):.4f}',
        })

layer_df = pd.DataFrame(layer_rows)
print('─' * 108)
print('PER-LAYER WEIGHT & BIAS STATISTICS')
print('─' * 108)
print(layer_df.to_string(index=False))

# ── Weight & bias distribution histograms ─────────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
colors = ['#e07b39', '#2196F3', '#4CAF50', '#9C27B0']

for row_i, ((label, model), color) in enumerate(zip(model_items, colors)):
    weights_all = np.concatenate([
        p.data.cpu().numpy().ravel()
        for n, p in model.named_parameters() if 'weight' in n
    ])
    biases_all = np.concatenate([
        p.data.cpu().numpy().ravel()
        for n, p in model.named_parameters() if 'bias' in n
    ])

    ax_w, ax_b = axes[row_i, 0], axes[row_i, 1]

    ax_w.hist(weights_all, bins=80, color=color, alpha=0.8, edgecolor='none')
    ax_w.axvline(weights_all.mean(), color='k', ls='--', lw=1.5,
                 label=f'μ={weights_all.mean():.4f}  σ={weights_all.std():.4f}')
    ax_w.axvline(0, color='red', ls=':', lw=1, alpha=0.5)
    ax_w.set_title(f'{label} — Weights ({weights_all.size:,} params)', fontweight='bold')
    ax_w.set_xlabel('Weight value'); ax_w.set_ylabel('Count')
    ax_w.legend(fontsize=8)

    ax_b.hist(biases_all, bins=40, color=color, alpha=0.8, edgecolor='none')
    ax_b.axvline(biases_all.mean(), color='k', ls='--', lw=1.5,
                 label=f'μ={biases_all.mean():.4f}  σ={biases_all.std():.4f}')
    ax_b.axvline(0, color='red', ls=':', lw=1, alpha=0.5)
    ax_b.set_title(f'{label} — Biases ({biases_all.size:,} params)', fontweight='bold')
    ax_b.set_xlabel('Bias value'); ax_b.set_ylabel('Count')
    ax_b.legend(fontsize=8)

plt.suptitle('Weight & Bias Distributions After Training', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('part1_weight_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ Saved part1_weight_distributions.png')

# ── Weight norm, bias norm & gradient norm evolution over training epochs ──────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors  = ['#e07b39', '#2196F3', '#4CAF50', '#9C27B0']
labels  = ['W2V 1ep', 'W2V 20ep', 'TF-IDF', 'GloVe']
hists   = [hist_b1, hist_b20, hist_c, hist_d]

for hist, label, color in zip(hists, labels, colors):
    x = range(1, len(hist['weight_norm']) + 1)
    axes[0].plot(x, hist['weight_norm'], label=label, color=color)
    axes[1].plot(x, hist['bias_norm'],   label=label, color=color)
    axes[2].plot(x, hist['grad_norm'],   label=label, color=color)

for ax, (title, ylabel) in zip(axes, [
    ('Weight L2 Norm over Epochs',   'L2 norm'),
    ('Bias L2 Norm over Epochs',     'L2 norm'),
    ('Gradient L2 Norm over Epochs', 'L2 norm'),
]):
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Parameter Norm Evolution During Training', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('part1_norm_evolution.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ Saved part1_norm_evolution.png')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ["#e07b39", "#2196F3", "#4CAF50", "#9C27B0"]

hists   = [hist_b1, hist_b20, hist_c, hist_d]
labels  = ["W2V 1ep", "W2V 20ep", "TF-IDF", "GloVe"]

for hist, label, color in zip(hists, labels, colors):
    axes[0].plot(hist["val_loss"], label=label, color=color)
    axes[1].plot(hist["val_acc"],  label=label, color=color)

for ax, title, ylabel in [
    (axes[0], "Validation loss by epoch",     "Cross-entropy loss"),
    (axes[1], "Validation accuracy by epoch", "Accuracy"),
]:
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("part1_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part1_curves.png  (include this in the report)")


---
## Confusion matrix — best model

The confusion matrix shows *which classes the best model confuses with which*.
Each row is the true class; each column is the predicted class.
The diagonal = correct predictions; off-diagonal cells = confusions.

In [ ]:
# ── Confusion matrix for the best-performing model ──────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import torch

# Identify best model by macro-F1
_all_results = [
    ('W2V 1ep',  model_b1,  X_te_1ep),
    ('W2V 20ep', model_b20, X_te_20ep),
    ('TF-IDF',   model_c,   X_te_tfidf),
    ('GloVe',    model_d,   X_te_glv),
]
_best_name, _best_model, _best_X = max(
    _all_results, key=lambda t: evaluate_on_test.__wrapped__(t[1], t[2], y_test)
    if hasattr(evaluate_on_test, '__wrapped__') else
    [res_b1, res_b20, res_c, res_d][['W2V 1ep','W2V 20ep','TF-IDF','GloVe'].index(t[0])]['f1']
)

# Use F1 dict already computed
_f1_map = {'W2V 1ep': res_b1['f1'], 'W2V 20ep': res_b20['f1'],
           'TF-IDF': res_c['f1'], 'GloVe': res_d['f1']}
_best_name = max(_f1_map, key=_f1_map.get)
_best_model_map = {'W2V 1ep': model_b1, 'W2V 20ep': model_b20,
                   'TF-IDF': model_c,    'GloVe': model_d}
_best_X_map = {'W2V 1ep': X_te_1ep, 'W2V 20ep': X_te_20ep,
               'TF-IDF': X_te_tfidf, 'GloVe': X_te_glv}
_best_model = _best_model_map[_best_name]
_best_X     = _best_X_map[_best_name]

_best_model.eval()
with torch.no_grad():
    _preds = _best_model(
        torch.tensor(_best_X, dtype=torch.float32).to(DEVICE)
    ).argmax(1).cpu().numpy()

cm = confusion_matrix(y_test, _preds, normalize='true')
fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=90, cmap='Blues', colorbar=False, values_format='.2f')
ax.set_title(f'Part 1 best model ({_best_name}) — normalised confusion matrix',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('part1_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Best model: {_best_name}  (macro-F1 = {_f1_map[_best_name]:.4f})')
print('→ Saved part1_confusion_matrix.png  (include in the report)')

In [ ]:
# ── Per-class F1 for the best model ─────────────────────────────────────────────
import pandas as pd

report = classification_report(y_test, _preds, target_names=class_names, output_dict=True)
per_class_df = pd.DataFrame({
    'class':    class_names,
    'f1_score': [report[c]['f1-score'] for c in class_names],
    'support':  [report[c]['support']  for c in class_names],
}).sort_values('f1_score')

print(per_class_df.to_string(index=False))
print(f"\nHardest classes: {per_class_df['class'].head(3).tolist()}")
print(f"Easiest classes: {per_class_df['class'].tail(3).tolist()}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#d32f2f' if f < 0.6 else '#FF9800' if f < 0.75 else '#4CAF50'
              for f in per_class_df['f1_score']]
ax.barh(per_class_df['class'], per_class_df['f1_score'], color=colors_bar)
ax.set_xlabel('F1 score', fontsize=12)
ax.set_title(f'Per-class F1 — {_best_name} (best Part-1 model)', fontsize=12, fontweight='bold')
ax.axvline(x=per_class_df['f1_score'].mean(), color='navy', linestyle='--', label='Macro-F1')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('part1_per_class_f1.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ Saved part1_per_class_f1.png  (include in the report)')

---
## ⚠️ Teardown — Run this before closing the notebook

> ### 🔴 STOP AND READ BEFORE YOU CLOSE
>
> **Google Colab charges per GPU-hour while the runtime is connected.**
> Even if you're not running cells, the runtime keeps billing.
>
> | Time forgotten | Approx cost (Colab Pro, T4 GPU) |
> |---|---|
> | 1 hour | ~$0.35 |
> | 1 day | ~$8 |
> | 1 week | ~$56 |
>
> **Run the two cells below, then disconnect your runtime.**

### What the teardown does
1. **Deletes model objects** from Python memory → frees GPU VRAM immediately
2. **Deletes checkpoint folders** from Colab disk → prevents disk-quota errors on next run
3. **Clears CUDA cache** → returns GPU memory to the OS
4. **Prints a checklist** → step-by-step confirmation you're fully shut down

### After running the teardown cells
Go to **Runtime → Disconnect and delete runtime** in the Colab menu bar.
That stops all billing for this session.

This information and practice was taken from a guest lecture regarding RAG.


In [ ]:
# ── Step 1: Delete model objects & free GPU VRAM ─────────────────────────────
import gc, torch

print('Freeing model objects from GPU memory…')

for _varname in ['model_b1', 'model_b20', 'model_c', 'model_d']:
    if _varname in globals():
        obj = globals()[_varname]
        if hasattr(obj, 'cpu'):
            obj.cpu()
        del globals()[_varname]
        print(f'  ✓ Deleted {_varname}')

for _varname in ['w2v_1ep', 'w2v_20ep', 'glove']:
    if _varname in globals():
        del globals()[_varname]
        print(f'  ✓ Deleted {_varname}')

for _varname in ['X_tr_tfidf', 'X_va_tfidf', 'X_te_tfidf',
                 'X_tr_1ep', 'X_tr_20ep', 'X_tr_glv',
                 'X_va_1ep', 'X_va_20ep', 'X_va_glv',
                 'X_te_1ep', 'X_te_20ep', 'X_te_glv']:
    if _varname in globals():
        del globals()[_varname]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e6
    print(f'CUDA cache cleared. GPU memory still allocated: {allocated:.1f} MB')
else:
    print('No GPU — RAM freed via gc.collect().')

print('✓ Step 1 complete.')


In [ ]:
# ── Step 2: Verify & checklist ────────────────────────────────────────────────
import shutil, os

print('=' * 65)
print('PART 1 TEARDOWN CHECKLIST')
print('=' * 65)

checks = [
    'Model objects (model_b1/b20/c/d) deleted from GPU memory',
    'Word2Vec and GloVe objects deleted',
    'NumPy feature matrices deleted',
    'GPU VRAM cleared (torch.cuda.empty_cache)',
]

for item in checks:
    print(f'  ✓ {item}')

for d in ['./part1_checkpoints']:
    if os.path.isdir(d):
        shutil.rmtree(d)
        print(f'  ✓ Deleted {d}')

print()
print('FINAL STEP (manual):')
print('  → Colab menu: Runtime → Disconnect and delete runtime')
print('  → Confirm at: https://colab.research.google.com/')
print('  → Verify no unexpected spend: https://console.cloud.google.com/billing')
print()
print('Part 1 teardown complete.')
